# Module 13 Instructor Colab Notebook: Graph Fundamentals and Connectivity

## A Mathematical Bridge to Artificial Intelligence

A table stores observations in rows and features in columns. A graph emphasizes something different: **relationships**. Its basic objects are vertices and edges, so it can represent people and friendships, web pages and hyperlinks, molecules and bonds, road intersections and streets, or entities and relations in a knowledge graph.

This module develops graph vocabulary carefully before moving to degree, paths, connectivity, graph isomorphism, knowledge graphs, and the neighborhood aggregation used by graph neural networks.

**Recommended pacing:** three class meetings. Have students draw each graph, encode it in two different representations, and explain each conclusion in words before relying on code.


> **Instructor emphasis**
>
> Graph terminology is easy to blur. Require students to say whether a graph is directed or undirected, whether weights are present, and whether repeated vertices or edges are permitted in a walk, trail, path, or cycle.

## Notebook map

| Section | Main mathematical ideas | Direct AI connection |
|---|---|---|
| 1 | Vertices, edges, graph types, representations | Relational data |
| 2 | Degree, in-degree, out-degree, strength | Local importance and activity |
| 3 | Walks, trails, paths, and cycles | Reasoning chains and routes |
| 4 | Connectivity and components | Reachability and data structure |
| 5 | Graph isomorphism | Structure matching |
| 6 | Knowledge graphs and GNN message passing | Retrieval and representation learning |
| Integrated lab | A small course-knowledge network | Combine representation, connectivity, and aggregation |


In [ ]:
# Core setup: compatible with Google Colab and standard Jupyter.
import math
import heapq
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

try:
    import ipywidgets as widgets
    from ipywidgets import interact, FloatSlider, IntSlider
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

print("Setup complete.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)


In [ ]:
def reveal_markdown(button_text, solution_markdown):
    if not WIDGETS_AVAILABLE:
        display(Markdown("**Instructor solution**"))
        display(Markdown(solution_markdown))
        return

    button = widgets.Button(description=button_text, button_style="info")
    output = widgets.Output()
    state = {"open": False}

    def toggle(_):
        with output:
            output.clear_output()
            state["open"] = not state["open"]
            if state["open"]:
                display(Markdown(solution_markdown))
                button.description = "Hide instructor solution"
            else:
                button.description = button_text

    button.on_click(toggle)
    display(button, output)


def draw_graph(vertices, edges, positions, directed=False, weights=None,
               title="Graph", highlighted_edges=None, highlighted_vertices=None):
    highlighted_edges = set(highlighted_edges or [])
    highlighted_vertices = set(highlighted_vertices or [])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.axis("off")
    for index, edge in enumerate(edges):
        u, v = edge
        color = "crimson" if (edge in highlighted_edges or (not directed and (v, u) in highlighted_edges)) else "#475569"
        arrowstyle = "->" if directed else "-"
        ax.annotate("", xy=positions[v], xytext=positions[u],
                    arrowprops=dict(arrowstyle=arrowstyle, linewidth=2.5,
                                    color=color, shrinkA=18, shrinkB=18))
        if weights is not None:
            midpoint = ((positions[u][0]+positions[v][0])/2,
                        (positions[u][1]+positions[v][1])/2)
            ax.text(midpoint[0], midpoint[1]+0.05, str(weights[index]),
                    ha="center", va="center",
                    bbox=dict(facecolor="white", edgecolor="none", pad=1))
    for vertex in vertices:
        color = "#fecaca" if vertex in highlighted_vertices else "#dbeafe"
        ax.text(positions[vertex][0], positions[vertex][1], str(vertex),
                ha="center", va="center", fontsize=12,
                bbox=dict(boxstyle="circle,pad=0.45", facecolor=color, edgecolor="#1d4ed8"))
    ax.set_title(title)
    plt.show()


## Module learning objectives

By the end of the module, students should be able to:

1. distinguish directed, undirected, weighted, and unweighted graphs;
2. convert between a drawing, an edge list, an adjacency list, and an adjacency matrix;
3. compute degrees and verify the handshaking identities;
4. distinguish walks, trails, paths, closed walks, and cycles;
5. determine connected components and directed reachability;
6. test graph-isomorphism claims using invariants and an explicit bijection; and
7. explain how knowledge graphs and graph neural networks use relational structure.


---

# Section 1: What Is a Graph?

## 1.1 From a collection of objects to a network

A **graph** is an ordered pair

$$
\boxed{G=(V,E),}
$$

where $$V$$ is a set of **vertices** and $$E$$ is a collection of **edges** connecting vertices. A vertex is also called a **node**. An edge is sometimes called a **link**.

For an undirected graph, an edge joining $$u$$ and $$v$$ can be written

$$
\{u,v\}.
$$

The braces communicate that order does not matter:

$$
\{u,v\}=\{v,u\}.
$$

For a directed graph, an edge is an ordered pair

$$
(u,v),
$$

also written $$u\to v.$$ Now the order matters because the arrow has a source and a destination.


## 1.2 Basic size vocabulary

The **order** of a finite graph is its number of vertices:

$$
|V|.
$$

The **size** of a finite graph is its number of edges:

$$
|E|.
$$

A **simple undirected graph** has no self-loops and no parallel edges. A **self-loop** joins a vertex to itself. **Parallel edges** are multiple edges joining the same endpoints. A graph that allows parallel edges is often called a **multigraph**.

In this module, graphs are simple unless stated otherwise. This assumption matters because some degree formulas count loops in a special way.


## 1.3 Major graph types

| Type | What an edge means | Example |
|---|---|---|
| Undirected | A symmetric relationship | Two songs share a performer |
| Directed | A relationship with orientation | One page links to another |
| Unweighted | Presence or absence is the main information | Two concepts are related |
| Weighted | An edge carries a number | Road distance or similarity score |

A graph can combine these descriptions. For example, a road network may be directed and weighted if one-way roads have travel times.

The number attached to a weighted edge is its **weight**. The meaning of a weight is application-dependent: a large value might mean strong similarity, high capacity, large cost, or long distance. Never interpret a weight before identifying what it measures.


In [ ]:
vertices13 = ["A", "B", "C", "D"]
edges13 = [("A", "B"), ("A", "C"), ("B", "C"), ("C", "D")]
positions13 = {"A": (0, 1), "B": (-1, 0), "C": (1, 0), "D": (1, -1.2)}

draw_graph(vertices13, edges13, positions13,
           title="An undirected graph with four vertices and four edges")


For the displayed graph,

$$
V=\{A,B,C,D\}
$$

and

$$
E=\{\{A,B\},\{A,C\},\{B,C\},\{C,D\}\}.
$$

Therefore,

$$
|V|=4
$$

and

$$
|E|=4.
$$


## 1.4 Four representations of the same graph

The drawing is visual, but algorithms need structured data.

1. An **edge list** records each edge.
2. An **adjacency list** records the neighbors of each vertex.
3. An **adjacency matrix** uses rows and columns indexed by vertices.
4. An **incidence matrix** uses vertices as rows and edges as columns.

For most of this module, we use the first three.

For a simple graph with vertex order $$A,B,C,D,$$ define

$$
A_{ij}=
\begin{cases}
1, & \text{if vertex }i\text{ is adjacent to vertex }j,\\
0, & \text{otherwise.}
\end{cases}
$$

The adjacency matrix of the displayed graph is

$$
\mathbf{A}=
\begin{bmatrix}
0&1&1&0\\
1&0&1&0\\
1&1&0&1\\
0&0&1&0
\end{bmatrix}.
$$

Because the graph is undirected, adjacency is symmetric, so

$$
\mathbf{A}^T=\mathbf{A}.
$$


In [ ]:
def adjacency_list(vertices, edges, directed=False):
    neighbors = {vertex: [] for vertex in vertices}
    for u, v in edges:
        neighbors[u].append(v)
        if not directed:
            neighbors[v].append(u)
    return {vertex: sorted(values) for vertex, values in neighbors.items()}


def adjacency_matrix(vertices, edges, directed=False, weights=None):
    if weights is not None and len(weights) != len(edges):
        raise ValueError("weights and edges must have the same length")
    index = {vertex: i for i, vertex in enumerate(vertices)}
    matrix = np.zeros((len(vertices), len(vertices)), dtype=float)
    for edge_index, (u, v) in enumerate(edges):
        value = 1 if weights is None else weights[edge_index]
        matrix[index[u], index[v]] = value
        if not directed:
            matrix[index[v], index[u]] = value
    return matrix


adjacency_list13 = adjacency_list(vertices13, edges13)
adjacency_matrix13 = adjacency_matrix(vertices13, edges13)

print("Adjacency list:", adjacency_list13)
print("Adjacency matrix:\n", adjacency_matrix13)
print("Symmetric?", np.allclose(adjacency_matrix13, adjacency_matrix13.T))


### Directed adjacency convention

For a directed graph, this notebook uses

$$
A_{ij}=1 \quad\text{when there is an edge }i\to j.
$$

Thus a row lists outgoing edges and a column lists incoming edges. Some books reverse this convention. Neither convention is inherently wrong, but a solution must state and use one convention consistently.

### A subtle weighted-graph issue

If an adjacency matrix stores zero for “no edge,” it cannot distinguish no edge from an edge whose genuine weight is zero. Software often stores an explicit edge list or a special missing value when zero-weight edges are possible.


## Classroom check 1

Let

$$
V=\{R,S,T\}
$$

and

$$
E=\{R\to S,\;S\to R,\;S\to T\}.
$$

1. Is the graph directed or undirected?
2. What are its order and size?
3. Write its adjacency list and adjacency matrix using the convention in this notebook.
4. Is the adjacency matrix symmetric?


In [ ]:
reveal_markdown("Open Section 1 solution", r"""
The graph is directed, with order $$3$$ and size $$3.$$

Its outgoing-neighbor adjacency list is

$$
R:[S],\qquad S:[R,T],\qquad T:[\;].
$$

With order $$R,S,T,$$

$$
\mathbf{A}=
\begin{bmatrix}
0&1&0\\
1&0&1\\
0&0&0
\end{bmatrix}.
$$

This matrix is not symmetric because $$S\to T$$ exists but $$T\to S$$ does not.
""")


---

# Section 2: Degree and Local Structure

## 2.1 Degree in an undirected graph

The **degree** of vertex $$v,$$ written $$\deg(v),$$ is the number of edges incident to it. Informally, it is the number of connections touching the vertex.

For the four-vertex graph,

$$
\deg(A)=2,\qquad \deg(B)=2,
$$

$$
\deg(C)=3,\qquad \deg(D)=1.
$$

An **isolated vertex** has degree $$0.$$ A **leaf** or **pendant vertex** has degree $$1.$$


## 2.2 The Handshaking Lemma

For every finite undirected graph,

$$
\boxed{\sum_{v\in V}\deg(v)=2|E|.}
$$

### Why it is true

Count edge endpoints in two ways.

- When degrees are added, each edge is counted once at each endpoint.
- Therefore each edge contributes exactly $$2$$ to the degree sum.

In our example,

$$
2+2+3+1=8
$$

and

$$
2|E|=2(4)=8.
$$

This argument is called **double counting**: the same collection of incidences is counted from two perspectives.


### Consequence: the number of odd-degree vertices is even

The degree sum $$2|E|$$ is even. The sum of all even degrees is even. Therefore the sum of the odd degrees must also be even. A sum of an odd number of odd integers is odd, so the graph must contain an even number of odd-degree vertices.

This is not merely a curiosity. It becomes the central condition for Eulerian trails in Module 14.


In [ ]:
degree13 = {vertex: len(adjacency_list13[vertex]) for vertex in vertices13}
degree_sum13 = sum(degree13.values())

print("Degrees:", degree13)
print("Sum of degrees:", degree_sum13)
print("Twice the number of edges:", 2 * len(edges13))
assert degree_sum13 == 2 * len(edges13)


## 2.3 In-degree and out-degree

An arrow has a direction, so a directed graph has two degree counts.

- The **out-degree** $$\deg^+(v)$$ counts edges leaving $$v.$$
- The **in-degree** $$\deg^-(v)$$ counts edges entering $$v.$$

Every directed edge leaves exactly one vertex and enters exactly one vertex. Consequently,

$$
\boxed{\sum_{v\in V}\deg^+(v)=|E|}
$$

and

$$
\boxed{\sum_{v\in V}\deg^-(v)=|E|.}
$$

For a web graph, high in-degree can indicate that many pages link to a page, while high out-degree means that the page links to many destinations. Degree is useful local information, but it is not a complete measure of importance.


## 2.4 Weighted degree or strength

For a weighted undirected graph, the **strength** of vertex $$v$$ is the sum of the weights on incident edges:

$$
s(v)=\sum_{u:\{u,v\}\in E}w_{uv}.
$$

Degree counts connections; strength totals their weights. A node can have low degree but high strength if it has a few very strong connections.

If weights represent costs rather than strengths, the word “strength” can be misleading. Always return to the application’s meaning.


## Classroom check 2

A directed graph has edges

$$
A\to B,\quad A\to C,\quad C\to A,\quad C\to B.
$$

1. Find every in-degree and out-degree.
2. Verify both directed degree-sum identities.
3. Which vertex has the greatest in-degree?


In [ ]:
reveal_markdown("Open Section 2 solution", r"""
The degree table is

| Vertex | In-degree | Out-degree |
|---|---:|---:|
| $$A$$ | 1 | 2 |
| $$B$$ | 2 | 0 |
| $$C$$ | 1 | 2 |

There are four edges, and

$$
1+2+1=4=|E|
$$

for in-degrees, while

$$
2+0+2=4=|E|
$$

for out-degrees. Vertex $$B$$ has the greatest in-degree.
""")


---

# Section 3: Walks, Trails, Paths, and Cycles

## 3.1 Why several terms are needed

A sequence of adjacent vertices can repeat vertices, repeat edges, or close back at its start. Graph theory gives these possibilities different names.

| Object | May repeat vertices? | May repeat edges? | Must end where it starts? |
|---|---:|---:|---:|
| Walk | Yes | Yes | No |
| Trail | Yes | No | No |
| Path | No | No | No |
| Closed walk | Yes | Yes | Yes |
| Cycle | Only the first/last vertex repeats | No | Yes |

A **walk** is a sequence

$$
v_0,v_1,\ldots,v_k
$$

in which every consecutive pair is joined by an edge. Its **length** is the number of edges traversed, namely $$k,$$ not the number of listed vertices.

A **trail** is a walk with no repeated edge. A **path** is a walk with no repeated vertex. Every path is a trail, and every trail is a walk, but the converses need not hold. Under the simple-undirected convention, a cycle has length at least three. In a directed graph, a two-vertex directed cycle can occur when both opposite arcs are present.


## 3.2 Detailed classification example

Use the graph with edges

$$
AB,\;AC,\;BC,\;CD.
$$

Consider the sequence

$$
A,B,C,A,C,D.
$$

Every consecutive pair is adjacent, so it is a walk. However, the edge $$AC$$ is traversed twice, first as $$C\to A$$ and then as $$A\to C.$$ Because the graph is undirected, these are the same edge. Therefore the walk is not a trail and not a path.

Now consider

$$
A,B,C,A.
$$

It starts and ends at $$A,$$ uses three distinct edges, and repeats no other vertex. It is a cycle of length $$3.$$

Finally,

$$
D,C,A,B
$$

is a path of length $$3.$$


In [ ]:
walk13 = ["D", "C", "A", "B"]
walk_edges13 = list(zip(walk13[:-1], walk13[1:]))

print("Vertex sequence:", walk13)
print("Traversed edges:", walk_edges13)
print("Length:", len(walk_edges13))

draw_graph(vertices13, edges13, positions13,
           title="Highlighted path D-C-A-B",
           highlighted_edges=walk_edges13,
           highlighted_vertices=walk13)


## 3.3 Directed paths and reasoning chains

In a directed graph, each step must follow an arrow. If a knowledge graph contains

$$
\text{Transformer}\to\text{Attention}
$$

and

$$
\text{Attention}\to\text{Similarity},
$$

then there is a directed path from Transformer to Similarity. The reverse sequence is not automatically a directed path.

This provides a simple model of a multi-hop reasoning chain: each edge supplies a permissible relation from one entity to the next. The existence of a path shows reachability, not necessarily that the inferred statement is logically valid; edge labels and semantics still matter.


## Classroom check 3

In the undirected graph with edges

$$
12,\;23,\;31,\;34,
$$

classify each sequence as precisely as possible.

1. $$1,2,3,1$$
2. $$1,2,3,4$$
3. $$1,2,3,1,3,4$$
4. What is the length of $$1,2,3,4$$?


In [ ]:
reveal_markdown("Open Section 3 solution", r"""
1. $$1,2,3,1$$ is a cycle of length $$3.$$
2. $$1,2,3,4$$ is a path and therefore also a trail and a walk.
3. $$1,2,3,1,3,4$$ is a walk. It is not a trail because edge $$13$$ is used twice, and it is not a path because vertices repeat.
4. The length is $$3$$ because three edges are traversed.
""")


---

# Section 4: Connectivity and Components

## 4.1 Connected undirected graphs

An undirected graph is **connected** if every pair of vertices is joined by a path. Connectivity is a global property: a vertex can have several local neighbors while the overall graph still has multiple separated pieces.

A **connected component** is a maximal connected subgraph. “Maximal” means it cannot be enlarged by adding another vertex from the graph while remaining connected.

If a graph has components

$$
\{A,B,C\},\qquad \{D,E\},\qquad \{F\},
$$

then it has three connected components. The isolated vertex $$F$$ forms a component by itself.


In [ ]:
def connected_components(vertices, edges):
    neighbors = adjacency_list(vertices, edges, directed=False)
    unseen = set(vertices)
    components = []
    while unseen:
        start = min(unseen, key=str)
        stack = [start]
        unseen.remove(start)
        component = []
        while stack:
            vertex = stack.pop()
            component.append(vertex)
            for neighbor in neighbors[vertex]:
                if neighbor in unseen:
                    unseen.remove(neighbor)
                    stack.append(neighbor)
        components.append(sorted(component, key=str))
    return components


component_vertices13 = list("ABCDEF")
component_edges13 = [("A", "B"), ("B", "C"), ("D", "E")]
components13 = connected_components(component_vertices13, component_edges13)
print("Connected components:", components13)


## 4.2 Bridges and articulation vertices

A **bridge** is an edge whose removal increases the number of connected components. An **articulation vertex** or **cut vertex** is a vertex whose removal, together with its incident edges, increases the number of connected components.

In the first graph of this notebook, edge $$CD$$ is a bridge. Removing it isolates $$D.$$ Vertex $$C$$ is an articulation vertex because removing $$C$$ separates $$D$$ from $$A$$ and $$B.$$

These concepts identify structural vulnerability. In a communication network, a bridge may be a single point of failure. In a concept graph, it may connect two otherwise separate topic clusters.


## 4.3 Weak and strong connectivity in directed graphs

Direction produces two common notions.

- A directed graph is **strongly connected** if for every pair $$u,v,$$ there is a directed path from $$u$$ to $$v$$ and a directed path from $$v$$ to $$u.$$
- It is **weakly connected** if replacing every arrow by an undirected edge produces a connected graph.

Every strongly connected directed graph is weakly connected. The converse is false.

For

$$
A\to B\to C,
$$

the underlying undirected graph is connected, so the directed graph is weakly connected. It is not strongly connected because there is no directed path from $$C$$ back to $$A.$$


## 4.4 Reachability matrices and powers of adjacency matrices

For a simple directed or undirected graph, the entry

$$
(\mathbf{A}^k)_{ij}
$$

counts the number of walks of length $$k$$ from vertex $$i$$ to vertex $$j,$$ under this notebook's row-to-column adjacency convention.

Why? Matrix multiplication sums over every possible intermediate vertex. For two steps,

$$
(\mathbf{A}^2)_{ij}=\sum_r A_{ir}A_{rj},
$$

and each nonzero product records a two-edge walk $$i\to r\to j.$$

This is a beautiful bridge between matrix multiplication and graph structure. Notice that it counts walks, not necessarily paths, because intermediate vertices may repeat for larger powers.


In [ ]:
A2_13 = adjacency_matrix13 @ adjacency_matrix13
print("A^2 counts two-step walks:\n", A2_13.astype(int))
print("Two-step walks from A to D:", int(A2_13[0, 3]))


There is exactly one two-step walk from $$A$$ to $$D,$$ namely

$$
A,C,D.
$$

The diagonal entry $$(\mathbf{A}^2)_{AA}=2$$ counts

$$
A,B,A
$$

and

$$
A,C,A.
$$

These are closed walks of length $$2,$$ not cycles in a simple undirected graph.


## Classroom check 4

For a graph with components $$\{1,2,3\},$$ $$\{4,5\},$$ and $$\{6\}:$$

1. Is the graph connected?
2. How many components are present?
3. Can any path run from $$2$$ to $$5$$?
4. If one edge is added to make $$\{1,2,3\}$$ meet $$\{4,5\},$$ how many components remain?


In [ ]:
reveal_markdown("Open Section 4 solution", r"""
The graph is not connected and has three components. No path can run from $$2$$ to $$5$$ because those vertices are in different components. One edge connecting the first two components merges them, leaving two components: the merged five-vertex component and isolated vertex $$6.$$
""")


---

# Section 5: Graph Isomorphism

## 5.1 Same structure, different labels

Two graphs

$$
G=(V_G,E_G)
$$

and

$$
H=(V_H,E_H)
$$

are **isomorphic** if there is a bijection

$$
f:V_G\to V_H
$$

such that adjacency is preserved:

$$
\{u,v\}\in E_G
\quad\Longleftrightarrow\quad
\{f(u),f(v)\}\in E_H.
$$

A **bijection** pairs every vertex of $$G$$ with exactly one vertex of $$H,$$ and uses every vertex of $$H$$ exactly once. Isomorphic graphs may look different on a page and use different labels, but their connection patterns are identical. For directed graphs, an isomorphism must preserve arrow orientation; for weighted or labeled graphs, it must also preserve the relevant weights or labels.


## 5.2 Invariants: fast ways to reject a claim

An **isomorphism invariant** is a property that relabeling cannot change. Isomorphic graphs must have the same:

- number of vertices;
- number of edges;
- sorted degree sequence;
- number of connected components;
- number of cycles of each length;
- pattern of adjacency among corresponding degree classes.

If any invariant differs, the graphs are not isomorphic. Matching a few invariants does **not** prove isomorphism; one must still provide an adjacency-preserving bijection or another complete argument.


## 5.3 Detailed isomorphism example

Let $$G$$ have edges

$$
12,\;23,\;34,\;41
$$

and let $$H$$ have edges

$$
ac,\;cb,\;bd,\;da.
$$

Both are four-cycles. Consider

$$
f(1)=a,\qquad f(2)=c,\qquad f(3)=b,\qquad f(4)=d.
$$

Check every edge:

$$
12\mapsto ac,
$$

$$
23\mapsto cb,
$$

$$
34\mapsto bd,
$$

$$
41\mapsto da.
$$

Each image is an edge of $$H.$$ Nonedges are also preserved because the mapping is a bijection between two graphs with exactly four edges. Therefore $$f$$ is an isomorphism.


## 5.4 Adjacency matrices under relabeling

Relabeling vertices rearranges rows and the matching columns of an adjacency matrix. If $$\mathbf{P}$$ is the permutation matrix encoding the new order, then the relabeled adjacency matrix has the form

$$
\boxed{\mathbf{A}_{\text{new}}=\mathbf{P}^T\mathbf{A}\mathbf{P}.}
$$

The same permutation must be applied to rows and columns because both axes list vertices. Isomorphism can therefore be expressed as simultaneous row-and-column permutation.


In [ ]:
A_cycle13 = np.array([
    [0, 1, 0, 1],
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [1, 0, 1, 0],
], dtype=int)

# New order: old vertices 1, 3, 2, 4 (zero-based indices 0, 2, 1, 3)
permutation13 = [0, 2, 1, 3]
P13 = np.eye(4, dtype=int)[:, permutation13]
A_relabel13 = P13.T @ A_cycle13 @ P13

print("Original adjacency matrix:\n", A_cycle13)
print("Relabeled adjacency matrix:\n", A_relabel13)
print("Degree multiset preserved:",
      sorted(A_cycle13.sum(axis=1)) == sorted(A_relabel13.sum(axis=1)))


## Classroom check 5

Graph $$G$$ has degree sequence

$$
(3,2,2,1).
$$

Graph $$H$$ has degree sequence

$$
(2,2,2,2).
$$

Can the graphs be isomorphic? Would matching degree sequences alone be enough to prove isomorphism?


In [ ]:
reveal_markdown("Open Section 5 solution", r"""
They cannot be isomorphic because the degree sequences differ, and degree sequence is an invariant. Matching degree sequences alone would not be enough to prove isomorphism; non-isomorphic graphs can share the same degree sequence. A full proof needs an adjacency-preserving bijection or an equivalent complete argument.
""")


---

# Section 6: Knowledge Graphs and Graph Neural Networks

## 6.1 Knowledge graphs

A **knowledge graph** represents entities as vertices and relationships as labeled, usually directed, edges. A fact is often stored as a triple

$$
(\text{subject},\text{relation},\text{object}).
$$

For example,

$$
(\text{cosine similarity},\text{uses},\text{inner product})
$$

can be drawn as

$$
\text{cosine similarity}\xrightarrow{\text{uses}}\text{inner product}.
$$

The edge label matters. “Uses,” “is a prerequisite for,” and “is an example of” are different relations even when they join the same two entities.


### Knowledge graphs and retrieval

In a retrieval system, graph structure can help locate related entities or follow multi-hop relations. A text-embedding search might retrieve semantically similar passages, while a knowledge graph can also expose explicit relationships.

The two representations are complementary:

| Embedding representation | Graph representation |
|---|---|
| A vector stores learned features | Vertices and edges store objects and relations |
| Similarity is often geometric | Relatedness is explicit in graph structure |
| Useful for approximate semantic retrieval | Useful for paths, constraints, and provenance |

Neither representation guarantees truth. The quality of retrieval still depends on the data, modeling choices, and verification process.


## 6.2 Node features and neighborhoods

Graph-learning problems often attach a feature vector

$$
\mathbf{h}_v\in\mathbb{R}^d
$$

to each vertex $$v.$$ The **neighborhood** of $$v,$$ written $$N(v),$$ is the set of vertices adjacent to it. For directed graphs, one must specify incoming, outgoing, or both kinds of neighbors.

A graph neural network updates a node representation by combining its current features with messages from neighbors. A generic message-passing layer is

$$
\mathbf{h}_v^{\text{new}}
=
\phi\left(
\mathbf{h}_v,
\operatorname{AGG}
\left\{
\psi(\mathbf{h}_u,e_{uv}):u\in N(v)
\right\}
\right).
$$

Here $$e_{uv}$$ denotes an optional edge feature, relation type, or label; $$\psi$$ constructs a message, $$\operatorname{AGG}$$ combines a set of messages, and $$\phi$$ produces the update.


## 6.3 A transparent one-dimensional message-passing example

Suppose node $$B$$ has feature $$4$$ and its neighbors $$A$$ and $$C$$ have features $$2$$ and $$8.$$ Use the mean of the node and its neighbors:

$$
h_B^{\text{new}}
=\frac{h_A+h_B+h_C}{3}.
$$

Substitute the values:

$$
h_B^{\text{new}}
=\frac{2+4+8}{3}
=\frac{14}{3}
\approx 4.6667.
$$

This is not a full trained GNN, but it exposes the central idea: the representation of a node changes using information from its local graph neighborhood.

An aggregator such as sum, mean, or maximum is **permutation invariant**: changing the order in which neighbors are listed does not change the result. This is essential because a graph's neighbor set has no natural first or second element.


In [ ]:
def mean_message_passing(features, neighbors, include_self=True):
    updated = {}
    for vertex, vertex_neighbors in neighbors.items():
        sources = list(vertex_neighbors)
        if include_self:
            sources.append(vertex)
        updated[vertex] = np.mean([features[source] for source in sources], axis=0)
    return updated


features_demo13 = {"A": 2.0, "B": 4.0, "C": 8.0}
neighbors_demo13 = {"A": ["B"], "B": ["A", "C"], "C": ["B"]}
updated_demo13 = mean_message_passing(features_demo13, neighbors_demo13)
print("Updated features:", updated_demo13)


### The same averaging operation in matrix form

Let $$A$$ be an adjacency matrix and add self-loops:

$$
\widetilde{A}=A+I.
$$

If $$\widetilde{D}$$ contains the row sums of $$\widetilde{A},$$ then self-inclusive mean aggregation is

$$
\boxed{H_{\mathrm{new}}=\widetilde{D}^{-1}\widetilde{A}H.}
$$

This equation connects neighborhood language to Module 2's matrix multiplication. More advanced graph convolutional networks learn transformations and often use symmetric normalization, but the transparent mean is the right first bridge.


In [ ]:
A_message13 = np.array([[0.0, 1.0, 0.0],
                        [1.0, 0.0, 1.0],
                        [0.0, 1.0, 0.0]])
A_tilde13 = A_message13+np.eye(3)
D_inverse13 = np.diag(1/A_tilde13.sum(axis=1))
H_message13 = np.array([[2.0], [4.0], [8.0]])
H_new_matrix13 = D_inverse13 @ A_tilde13 @ H_message13
print("Matrix-form updated features:\n", H_new_matrix13.ravel())
print("Dictionary-form updated features:", updated_demo13)


### Important limitation: repeated averaging can oversmooth

If neighbor averaging is repeated many times on a connected graph, node features can become increasingly similar. This phenomenon is called **oversmoothing**. Practical GNNs use learned transformations, nonlinearities, residual connections, attention, normalization, or limited depth to preserve useful distinctions.

The graph alone is also not enough. Node features, edge features, direction, labels, training targets, and sampling choices all influence what the model can learn.


---

# Integrated Lab: A Course-Knowledge Network

Consider six topics:

$$
V=\{\text{Vectors},\text{Matrices},\text{PCA},\text{Probability},\text{Bayes},\text{Graphs}\}.
$$

Create an undirected “strongly related” graph with edges

$$
\{\text{Vectors},\text{Matrices}\},
$$

$$
\{\text{Matrices},\text{PCA}\},
$$

$$
\{\text{Vectors},\text{PCA}\},
$$

$$
\{\text{Probability},\text{Bayes}\}.
$$

Graphs is initially isolated. This is a simplified teaching network, not a claim that graph theory is unrelated to the other subjects.


In [ ]:
course_vertices13 = ["Vectors", "Matrices", "PCA", "Probability", "Bayes", "Graphs"]
course_edges13 = [
    ("Vectors", "Matrices"),
    ("Matrices", "PCA"),
    ("Vectors", "PCA"),
    ("Probability", "Bayes"),
]
course_positions13 = {
    "Vectors": (-1.2, 0.9), "Matrices": (-2.0, -0.3), "PCA": (-0.4, -0.3),
    "Probability": (1.0, 0.7), "Bayes": (2.0, 0.0), "Graphs": (1.2, -1.0),
}

course_adjacency13 = adjacency_list(course_vertices13, course_edges13)
course_matrix13 = adjacency_matrix(course_vertices13, course_edges13)
course_degrees13 = {v: len(course_adjacency13[v]) for v in course_vertices13}
course_components13 = connected_components(course_vertices13, course_edges13)

draw_graph(course_vertices13, course_edges13, course_positions13,
           title="Simplified course-knowledge network")
print("Degrees:", course_degrees13)
print("Components:", course_components13)
print("Adjacency matrix:\n", course_matrix13.astype(int))


## Integrated-lab tasks

1. Verify the handshaking lemma.
2. Identify every component and every cycle.
3. Add the edge $$\{\text{Graphs},\text{Matrices}\}.$$ How many components remain?
4. Add the edge $$\{\text{Graphs},\text{Probability}\}.$$ Is the resulting graph connected?
5. Assign the scalar features

$$
1,2,3,4,5,6
$$

in the displayed vertex order. Perform one self-inclusive mean-aggregation step on the original graph.
6. Explain why the updated feature for the isolated Graphs vertex does not change.


In [ ]:
course_features13 = {
    vertex: float(index + 1) for index, vertex in enumerate(course_vertices13)
}
course_updated13 = mean_message_passing(
    course_features13, course_adjacency13, include_self=True
)

print("Original features:", course_features13)
print("After one mean-aggregation step:", course_updated13)


In [ ]:
reveal_markdown("Open integrated-lab solution", r"""
The degree sum is

$$
2+2+2+1+1+0=8,
$$

while

$$
2|E|=2(4)=8.
$$

The components are

$$
\{\text{Vectors},\text{Matrices},\text{PCA}\},
$$

$$
\{\text{Probability},\text{Bayes}\},
$$

and

$$
\{\text{Graphs}\}.
$$

The first component contains one three-cycle. Adding Graphs--Matrices merges Graphs with the first component, leaving two components. Adding Graphs--Probability then joins those two components, so the graph becomes connected.

The original self-inclusive mean updates are

$$
h_{\text{Vectors}}'=\frac{1+2+3}{3}=2,
$$

$$
h_{\text{Matrices}}'=\frac{2+1+3}{3}=2,
$$

$$
h_{\text{PCA}}'=\frac{3+1+2}{3}=2,
$$

$$
h_{\text{Probability}}'=\frac{4+5}{2}=4.5,
$$

$$
h_{\text{Bayes}}'=\frac{5+4}{2}=4.5,
$$

and

$$
h_{\text{Graphs}}'=6.
$$

The isolated node has no neighbor messages. Because the update includes the node itself, its mean contains only its own value and remains $$6.$$
""")


---

# Terminology Reference

| Term | Precise meaning |
|---|---|
| Graph | A vertex set together with an edge collection |
| Vertex/node | An object represented in a graph |
| Edge/link | A relationship joining vertices |
| Order and size | Numbers of vertices and edges |
| Directed edge | An ordered relationship $$u\to v$$ |
| Weight | A numerical edge attribute whose meaning is application-dependent |
| Adjacency | The relation of being joined by an edge |
| Degree | Number of incident edges in an undirected graph |
| In-degree/out-degree | Counts of entering/leaving directed edges |
| Walk/trail/path | Successively stricter vertex sequences connected by edges |
| Cycle | A closed route with no repeated vertex except its start/end |
| Component | A maximal connected subgraph |
| Strong connectivity | Mutual directed reachability for every pair |
| Isomorphism | An adjacency-preserving bijection between vertex sets |
| Knowledge graph | Entities connected by labeled semantic relations |
| Message passing | Updating node features from neighborhood information |


# Common Mistakes and Corrections

1. **Mistake:** Treating $$u\to v$$ as equivalent to $$v\to u.$$
   **Correction:** That equivalence holds only for an undirected edge.
2. **Mistake:** Calling every vertex sequence a path.
   **Correction:** A path repeats no vertex; use walk when repetition is allowed.
3. **Mistake:** Saying a graph with one high-degree vertex is connected.
   **Correction:** Connectivity requires a path between every pair of vertices.
4. **Mistake:** Concluding isomorphism from a matching degree sequence.
   **Correction:** Matching invariants are necessary but may not be sufficient.
5. **Mistake:** Treating a missing weighted edge as automatically identical to a zero-weight edge.
   **Correction:** The data representation must preserve that distinction when zero is a valid weight.
6. **Mistake:** Assuming graph-neighbor order carries meaning.
   **Correction:** A graph neighborhood is a set; GNN aggregation should respect permutation invariance.
7. **Mistake:** Treating a graph relationship as proof of truth or causality.
   **Correction:** Edges encode the supplied relationship data, whose provenance and meaning must be evaluated.


---

# Module 13 Summary

1. A graph models objects as vertices and relationships as edges.
2. Direction and weight change both interpretation and representation.
3. Adjacency lists are often sparse and convenient; adjacency matrices connect graphs to linear algebra.
4. Degree measures local connectivity, and degree sums obey double-counting identities.
5. Walks, trails, paths, and cycles differ by which repetitions they allow.
6. Components describe global connectivity; directed graphs distinguish weak and strong connectivity.
7. Isomorphic graphs have identical structure after relabeling.
8. Knowledge graphs encode explicit relations, while GNNs update features using neighborhood messages.

## Exit ticket

1. Why is an undirected adjacency matrix symmetric?
2. What does the Handshaking Lemma count twice?
3. Give a trail that is not a path.
4. What is the difference between weak and strong connectivity?
5. Why does a matching degree sequence fail to prove isomorphism?
6. Why should a GNN aggregator be permutation invariant?
